# **NMT**


In [ ]:
# ============================================================
# FINAL LIGHT TOY UNSUPERVISED NMT MODEL
# EN <-> IT
# Different non-parallel monolingual corpora
# Monolingual Warm-up + DAE + Light Back-Translation
# ============================================================

# Colab:
!pip install datasets sacrebleu -q

from datasets import load_dataset
import sacrebleu
import pandas as pd
import torch
import torch.nn as nn
import random, re, math, time
from collections import Counter

# ============================================================
# SETTINGS
# ============================================================

# Different ranges are used for English and Italian training.
# Therefore, the training sentences are NOT translations of each other.
EN_TRAIN_SPLIT = "train[1000:11000]"
IT_TRAIN_SPLIT = "train[20000:30000]"

SIZES = [50, 100, 250, 500]

MAX_LEN = 30
VOCAB_SIZE = 10000
BATCH_SIZE = 32

EPOCHS_INIT = 3
EPOCHS_DAE = 3
EPOCHS_BT = 1
BT_SENTENCES = 1500

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

start_time = time.time()

# ============================================================
# TOKENIZATION
# ============================================================

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text

# ============================================================
# DATA — DIFFERENT NON-PARALLEL CORPORA
# ============================================================

# We load two different slices from OPUS Books.
# Only English is kept from the first slice.
# Only Italian is kept from the second slice.
# No aligned EN–IT pair is used during training.

en_data = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-it",
    split=EN_TRAIN_SPLIT
)

it_data = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-it",
    split=IT_TRAIN_SPLIT
)

en_train = [x["translation"]["en"] for x in en_data]
it_train = [x["translation"]["it"] for x in it_data]

# Shuffle independently so that even the training order is unrelated.
random.shuffle(en_train)
random.shuffle(it_train)

print("English monolingual sentences:", len(en_train))
print("Italian monolingual sentences:", len(it_train))
print("Parallel training pairs used: 0")

# ============================================================
# SHARED VOCABULARY
# ============================================================

special_tokens = [
    "<pad>", "<unk>", "<eos>",
    "<en>", "<it>",
    "<bos_en>", "<bos_it>"
]

counter = Counter()

# One shared vocabulary is built from both monolingual corpora.
for s in en_train:
    counter.update(tokenize(s)[:MAX_LEN])

for s in it_train:
    counter.update(tokenize(s)[:MAX_LEN])

vocab_words = [
    w for w, _ in counter.most_common(
        VOCAB_SIZE - len(special_tokens)
    )
]

itos = special_tokens + vocab_words
stoi = {w: i for i, w in enumerate(itos)}

PAD = stoi["<pad>"]
UNK = stoi["<unk>"]
EOS = stoi["<eos>"]

def encode_sentence(text, lang_token):
    tokens = tokenize(text)[:MAX_LEN]
    return (
        [stoi[lang_token]]
        + [stoi.get(t, UNK) for t in tokens]
        + [EOS]
    )

def encode_target(text, bos_token):
    tokens = tokenize(text)[:MAX_LEN]
    return (
        [stoi[bos_token]]
        + [stoi.get(t, UNK) for t in tokens]
        + [EOS]
    )

def decode_ids(ids):
    tokens = []

    for i in ids:
        tok = itos[int(i)]

        if tok in special_tokens:
            continue

        tokens.append(tok)

    return detokenize(tokens)

def pad_batch(batch):
    max_len = max(len(x) for x in batch)

    return torch.tensor(
        [
            x + [PAD] * (max_len - len(x))
            for x in batch
        ],
        dtype=torch.long
    )

# ============================================================
# DENOISING
# ============================================================

def corrupt_sentence(text):
    tokens = tokenize(text)[:MAX_LEN]

    # Random token deletion
    tokens = [
        t for t in tokens
        if random.random() > 0.15
    ]

    # Small local word-order noise
    if len(tokens) > 3:
        i = random.randint(0, len(tokens) - 2)
        tokens[i], tokens[i + 1] = tokens[i + 1], tokens[i]

    # Avoid an empty sentence
    if not tokens:
        tokens = ["<unk>"]

    return detokenize(tokens)

# ============================================================
# MODEL
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class ToyTransformerNMT(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=192,
        nhead=4,
        num_layers=2
    ):
        super().__init__()

        # One shared embedding table for both languages
        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD
        )

        self.pos = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=384,
            dropout=0.1,
            batch_first=True
        )

        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt_in):
        src_pad = src.eq(PAD)
        tgt_pad = tgt_in.eq(PAD)

        tgt_len = tgt_in.size(1)

        causal_mask = (
            nn.Transformer.generate_square_subsequent_mask(tgt_len)
            .to(src.device)
        )

        src_emb = self.pos(self.embedding(src))
        tgt_emb = self.pos(self.embedding(tgt_in))

        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=causal_mask,
            src_key_padding_mask=src_pad,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad
        )

        return self.output(out)

model = ToyTransformerNMT(len(itos)).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4
)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD
)

# ============================================================
# TRAINING
# ============================================================

def compute_batch_loss(
    src_texts,
    tgt_texts,
    src_lang,
    tgt_bos
):
    """
    Compute a differentiable loss tensor.
    No optimizer step is performed here, so several losses can be
    combined before one joint update.
    """
    model.train()

    src_ids = [
        encode_sentence(s, src_lang)
        for s in src_texts
    ]

    tgt_ids = [
        encode_target(t, tgt_bos)
        for t in tgt_texts
    ]

    src = pad_batch(src_ids).to(DEVICE)
    tgt = pad_batch(tgt_ids).to(DEVICE)

    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]

    logits = model(src, tgt_in)

    return criterion(
        logits.reshape(-1, logits.size(-1)),
        tgt_out.reshape(-1)
    )


def optimize_loss(loss):
    """Apply one optimizer update from a supplied total loss."""
    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()

    return loss.item()

# ============================================================
# TRANSLATION
# ============================================================

@torch.no_grad()
def translate_one(
    sentence,
    src_lang="<en>",
    tgt_bos="<bos_it>",
    max_len=30
):
    model.eval()

    src = pad_batch([
        encode_sentence(sentence, src_lang)
    ]).to(DEVICE)

    generated = torch.tensor(
        [[stoi[tgt_bos]]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_len):
        logits = model(src, generated)

        next_id = (
            logits[:, -1, :]
            .argmax(dim=-1)
            .item()
        )

        next_token = torch.tensor(
            [[next_id]],
            dtype=torch.long,
            device=DEVICE
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

        if next_id == EOS:
            break

    return decode_ids(
        generated[0].cpu().tolist()
    )

def translate_list(
    sentences,
    src_lang="<en>",
    tgt_bos="<bos_it>"
):
    return [
        translate_one(
            s,
            src_lang=src_lang,
            tgt_bos=tgt_bos
        )
        for s in sentences
    ]

# ============================================================
# LEVEL 1 — MONOLINGUAL AUTOENCODER WARM-UP
# ============================================================

# IMPORTANT:
# The original code used true EN–IT pairs here.
# That was supervised training.
#
# This modified version uses only:
#     clean English -> clean English
#     clean Italian -> clean Italian
#
# Therefore no parallel sentence pairs are used.

print("\nLEVEL 1 — Monolingual autoencoder warm-up")

for epoch in range(EPOCHS_INIT):
    losses = []

    max_train_len = max(
        len(en_train),
        len(it_train)
    )

    for i in range(0, max_train_len, BATCH_SIZE):
        batch_losses = []

        en_batch = en_train[i:i + BATCH_SIZE]
        it_batch = it_train[i:i + BATCH_SIZE]

        if en_batch:
            loss_en = compute_batch_loss(
                en_batch,
                en_batch,
                "<en>",
                "<bos_en>"
            )
            batch_losses.append(loss_en)

        if it_batch:
            loss_it = compute_batch_loss(
                it_batch,
                it_batch,
                "<it>",
                "<bos_it>"
            )
            batch_losses.append(loss_it)

        if batch_losses:
            total_init_loss = sum(batch_losses) / len(batch_losses)
            losses.append(optimize_loss(total_init_loss))

    print(
        f"INIT epoch {epoch + 1}/{EPOCHS_INIT}, "
        f"loss = {sum(losses) / len(losses):.4f}"
    )

# ============================================================
# LEVEL 2 + LEVEL 3 — JOINT DENOISING + BACK-TRANSLATION
# ============================================================

print("\nLEVEL 2 + LEVEL 3 — Joint DAE and bidirectional BT")

JOINT_EPOCHS = max(EPOCHS_DAE, EPOCHS_BT)

bt_count = min(
    BT_SENTENCES,
    len(en_train),
    len(it_train)
)

for epoch in range(JOINT_EPOCHS):
    losses = []

    # Use the same paired iteration structure:
    # one English monolingual batch + one Italian monolingual batch
    joint_count = min(len(en_train), len(it_train))

    for i in range(0, joint_count, BATCH_SIZE):
        en_batch = en_train[i:i + BATCH_SIZE]
        it_batch = it_train[i:i + BATCH_SIZE]

        component_losses = []

        # ----------------------------------------------------
        # LEVEL 2A — English denoising loss
        # ----------------------------------------------------
        if en_batch:
            noisy_en = [
                corrupt_sentence(s)
                for s in en_batch
            ]

            loss_lm_en = compute_batch_loss(
                noisy_en,
                en_batch,
                "<en>",
                "<bos_en>"
            )

            component_losses.append(loss_lm_en)

        # ----------------------------------------------------
        # LEVEL 2B — Italian denoising loss
        # ----------------------------------------------------
        if it_batch:
            noisy_it = [
                corrupt_sentence(s)
                for s in it_batch
            ]

            loss_lm_it = compute_batch_loss(
                noisy_it,
                it_batch,
                "<it>",
                "<bos_it>"
            )

            component_losses.append(loss_lm_it)

        # ----------------------------------------------------
        # LEVEL 3A — EN -> synthetic IT -> reconstruct EN
        # ----------------------------------------------------
        if en_batch and i < bt_count:
            fake_it = translate_list(
                en_batch,
                src_lang="<en>",
                tgt_bos="<bos_it>"
            )

            loss_bt_en = compute_batch_loss(
                fake_it,
                en_batch,
                "<it>",
                "<bos_en>"
            )

            component_losses.append(loss_bt_en)

        # ----------------------------------------------------
        # LEVEL 3B — IT -> synthetic EN -> reconstruct IT
        # ----------------------------------------------------
        if it_batch and i < bt_count:
            fake_en = translate_list(
                it_batch,
                src_lang="<it>",
                tgt_bos="<bos_en>"
            )

            loss_bt_it = compute_batch_loss(
                fake_en,
                it_batch,
                "<en>",
                "<bos_it>"
            )

            component_losses.append(loss_bt_it)

        # ----------------------------------------------------
        # ONE JOINT UPDATE
        # L_total = LM_EN + LM_IT + BT_EN + BT_IT
        # ----------------------------------------------------
        if component_losses:
            total_loss = sum(component_losses) / len(component_losses)
            losses.append(optimize_loss(total_loss))

    print(
        f"JOINT epoch {epoch + 1}/{JOINT_EPOCHS}, "
        f"loss = {sum(losses) / len(losses):.4f}"
    )

# ============================================================
# EVALUATION
# ============================================================

# Parallel references are used ONLY for evaluation.
# They are never used for training.

def evaluate_nmt(sizes):
    results = []

    for n in sizes:
        print(
            f"\nEvaluating EN→IT "
            f"on first {n} held-out sentences..."
        )

        test_data = load_dataset(
            "Helsinki-NLP/opus_books",
            "en-it",
            split=f"train[:{n}]"
        )

        sources = [
            x["translation"]["en"]
            for x in test_data
        ]

        references = [
            x["translation"]["it"]
            for x in test_data
        ]

        hypotheses = translate_list(
            sources,
            src_lang="<en>",
            tgt_bos="<bos_it>"
        )

        bleu = sacrebleu.corpus_bleu(
            hypotheses,
            [references]
        )

        print(f"BLEU = {bleu.score:.2f}")

        results.append({
            "Number of Sentences": n,
            "Non-Parallel Toy NMT BLEU": round(
                bleu.score,
                2
            )
        })

    return pd.DataFrame(results)

nmt_results = evaluate_nmt(SIZES)

print("\nFinal Non-Parallel Toy NMT Results:")
display(nmt_results)

# ============================================================
# RUNTIME
# ============================================================

end_time = time.time()
elapsed = end_time - start_time

print("\nRuntime Summary")
print("----------------")
print("Device used:", DEVICE)
print(f"Total time: {elapsed:.2f} seconds")
print(f"Total time: {elapsed / 60:.2f} minutes")

Device: cuda
English monolingual sentences: 10000
Italian monolingual sentences: 10000
Parallel training pairs used: 0

LEVEL 1 — Monolingual autoencoder warm-up


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


INIT epoch 1/3, loss = 5.0277
INIT epoch 2/3, loss = 2.5264
INIT epoch 3/3, loss = 1.2932

LEVEL 2 + LEVEL 3 — Joint DAE and bidirectional BT
JOINT epoch 1/3, loss = 2.8682
JOINT epoch 2/3, loss = 2.2929
JOINT epoch 3/3, loss = 2.0163

Evaluating EN→IT on first 50 held-out sentences...
BLEU = 0.07

Evaluating EN→IT on first 100 held-out sentences...
BLEU = 0.12

Evaluating EN→IT on first 250 held-out sentences...
BLEU = 0.08

Evaluating EN→IT on first 500 held-out sentences...
BLEU = 0.06

Final Non-Parallel Toy NMT Results:


,Number of Sentences,Non-Parallel Toy NMT BLEU
0,50,0.07
1,100,0.12
2,250,0.08
3,500,0.06



Runtime Summary
----------------
Device used: cuda
Total time: 1106.81 seconds
Total time: 18.45 minutes
